In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# פונקציית Circuit של IBM

> **Note:** * פונקציות Qiskit הן תכונה ניסיונית הזמינה רק למשתמשי IBM Quantum&reg; Premium Plan,‏ Flex Plan ו-On-Prem (דרך IBM Quantum Platform API) Plan. הן נמצאות בסטטוס שחרור תצוגה מקדימה וכפופות לשינויים.

## סקירה כללית
פונקציית Circuit של IBM&reg; מקבלת [PUBs מופשטים](./primitive-input-output) כקלט ומחזירה ערכי ציפייה מותאמים כפלט. פונקציית Circuit זו כוללת צינור עיבוד אוטומטי ומותאם אישית כדי לאפשר לחוקרים להתמקד בגילוי אלגוריתמים ויישומים.

## תיאור
לאחר שתגיש את ה-PUB שלך, המעגלים המופשטים והאובזרבבלים שלך יעברו טרנספילציה אוטומטית, הרצה על חומרה ועיבוד-לאחר כדי להחזיר ערכי ציפייה מותאמים. לשם כך, הפונקציה משלבת את הכלים הבאים:

- [שירות Qiskit Transpiler](./qiskit-transpiler-service), כולל בחירה אוטומטית של מעברי טרנספילציה מונחי בינה מלאכותית והיוריסטיים לתרגום המעגלים המופשטים שלך למעגלי ISA מותאמי חומרה
- [טכניקות דיכוי ומיתון שגיאות הנדרשות לחישוב בקנה מידה שימושי](./error-mitigation-and-suppression-techniques), כולל twirling של מדידות ושערים, ניתוק דינמי, Twirled Readout Error eXtinction‏ (TREX),‏ Zero-Noise Extrapolation‏ (ZNE) ו-Probabilistic Error Amplification‏ (PEA)
- [Qiskit Runtime Estimator](./get-started-with-primitives), להרצת PUBs מסוג ISA על חומרה והחזרת ערכי ציפייה מותאמים

![IBM Circuit function](../docs/images/guides/ibm-circuit-function/ibm-circuit-function.svg)

## תחילת עבודה
אמת את זהותך באמצעות [מפתח ה-API](http://quantum.cloud.ibm.com/) שלך ובחר את פונקציית Qiskit כדלקמן. (קטע קוד זה מניח שכבר [שמרת את חשבונך](/guides/functions#install-qiskit-functions-catalog-client) בסביבה המקומית שלך.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

function = catalog.load("ibm/circuit-function")

## דוגמה
כדי להתחיל, נסה את הדוגמה הבסיסית הזו:

In [2]:
from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import QiskitRuntimeService

# You can skip this step if you have a target backend, e.g.
# backend_name = "ibm_brisbane"
# You'll need to specify the credentials when initializing QiskitRuntimeService, if they were not previously saved.
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits
pubs = [(circuit, observable)]

job = function.run(
    # Use `backend_name=backend_name` if you didn't initialize a backend object
    backend_name=backend.name,
    pubs=pubs,
)

בדוק את [הסטטוס](/guides/functions#check-job-status) של עומס העבודה של פונקציית Qiskit שלך או אחזר את [התוצאות](/guides/functions#retrieve-results) כדלקמן:

In [3]:
print(job.status())
result = job.result()

QUEUED


The results have the same format as an [Estimator result](/docs/guides/primitive-input-output#estimator-output):

In [4]:
print(f"The result of the submitted job had {len(result)} PUB\n")
print(
    f"The associated PubResult of this job has the following DataBins:\n {result[0].data}\n"
)
print(f"And this DataBin has attributes: {result[0].data.keys()}")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)

The result of the submitted job had 1 PUB

The associated PubResult of this job has the following DataBins:
 DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>))

And this DataBin has attributes: dict_keys(['evs', 'stds', 'ensemble_standard_error'])
The expectation values measured from this PUB are: 
1.02116704805492


לתוצאות יש את אותו פורמט כמו [תוצאת Estimator](/guides/primitive-input-output#estimator-output):

In [5]:
options = {"mitigation_level": 2}

job = function.run(backend_name=backend.name, pubs=pubs, options=options)

#### All available options

In addition to `mitigation_level`, the IBM Circuit function also provides a select number of advanced options that allow you to fine-tune the cost/accuracy trade-off. The following table shows all the available options:

| Option | Sub-option | Sub-sub-option | Description | Choices | Default |
|-|-|-|-|-|-|
| default_precision |  |  | The default precision to use for any PUB or `run()`<br/>call that does not specify one.<br/>Each input PUB can specify its own precision. If the `run()` method is given a precision, then that value is used for all PUBs in the `run()` call that do not specify their own.  | float > 0 | 0.015625 |
| max_execution_time |  |  | Maximum execution time in seconds, which is based<br/>on QPU usage (not wall clock time). QPU usage is the<br/>amount of time that the QPU is dedicated to processing your job. If a job exceeds this time limit, it is forcibly canceled. | Integer number of seconds in the range [1, 10800] |  |
| mitigation_level |  |  | How much error suppression and mitigation to apply. Refer to the [Mitigation level](#mitigation-level) section for more information about the methods used at each level. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | How much optimization to perform on the circuits. [Higher levels](/docs/guides/set-optimization) generate more optimized circuits, at the expense of longer transpilation time. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | Whether to enable dynamical decoupling. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) for the explanation of the method.  | True/False | True |
|  | sequence_type |  | Which dynamical decoupling sequence to use.<br/>* `XX`: use the sequence `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: use the sequence `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: use the sequence<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | Whether to apply 2-qubit Clifford gate twirling. | True/False | False |
|  | enable_measure |  | Whether to enable twirling of measurements. | True/False | True |
| resilience | measure_mitigation |  | Whether to enable TREX measurement error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) for the explanation of the method.  | True/False | True |
|  | zne_mitigation |  | Whether to turn on Zero Noise Extrapolation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) for the explanation of the method.  | True/False | False |
|  | zne | amplifier | Which technique to use for amplifying noise. One of: <br/> - `gate_folding` (default) uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are chosen randomly.<br/><br/> - `gate_folding_front` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the front of the topologically ordered DAG circuit.<br/><br/> - `gate_folding_back` uses 2-qubit gate folding to amplify noise. If the noise factor requires amplifying only a subset of the gates, then these gates are selected from the back of the topologically ordered DAG circuit.<br/><br/> - `pea` uses a technique called Probabilistic error amplification (PEA) to amplify noise. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) for the explanation of the method.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | Noise factors to use for noise amplification. | list of floats; each float >= 1 | (1, 1.5, 2) for PEA, and (1, 3, 5) otherwise. |
|  |  | extrapolator | Noise factors to evaluate the fit extrapolation models at. This option does not affect execution or model fitting in any way; it only determines the points at which the `extrapolator` objects are evaluated to be returned in the data fields called `evs_extrapolated` and `stds_extrapolated`. | one or more of `exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | Whether to turn on Probabilistic Error Cancellation error mitigation method. Refer to [Error suppression and mitigation techniques](/docs/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) for the explanation of the method.  | True/False | False |
|  | pec | max_overhead | The maximum circuit sampling overhead allowed, or `None` for no maximum. | None/ integer >1 | 100 |

In the following example, setting the mitigation level to 1 initially turns off ZNE mitigation, but setting `zne_mitigation` to `True` overrides the relevant setup from `mitigation_level`.

In [6]:
options = {"mitigation_level": 1, "resilience": {"zne_mitigation": True}}

## קלטים
ראה את הטבלה הבאה לכל פרמטרי הקלט שפונקציית Circuit של IBM מקבלת. חלק ה-[אפשרויות](#options) שלאחר מכן מפרט יותר לגבי האפשרויות הזמינות ב-`options`.

| שם | סוג | תיאור | נדרש | ברירת מחדל | דוגמה |
|-----------|----------------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------|--------------------------------------------------------------------------|------------------------------------------|
| backend_name   | str                        | שם ה-Backend לביצוע השאילתה.                                                                                                                                                                                                              | כן      | N/A                                                                      | `ibm_fez`                                |
| pubs      | Iterable[EstimatorPubLike] | אוסף ניתן לחזרה של אובייקטי PUB-like מופשטים (primitive unified bloc), כגון טאפלים `(circuit, observables)` או `(circuit, observables, parameter_values)`. ראה [סקירת PUBs](/guides/primitive-input-output#overview-of-pubs) למידע נוסף. המעגלים יכולים להיות מופשטים (non-ISA). | כן      | N/A                                                                      | (circuit, observables, parameter_values) |
| options   | dict                       | אפשרויות קלט. ראה את סעיף **אפשרויות** לפרטים נוספים.                                                                                                                                                                                                | לא       | ראה את סעיף **אפשרויות** לפרטים.                                                   | `{"optimization_level": 3}`                |
| instance  | str                        | שם המשאב בענן של המופע לשימוש בפורמט זה.                                                                                                                                                                                                        | לא       | נבחר אחד אקראי אם לחשבונך יש גישה למספר מופעים. | `CRN`                   |

### אפשרויות
#### מבנה
בדומה לפרימיטיביות של Qiskit Runtime, אפשרויות לפונקציית Circuit של IBM ניתן לציין כמילון מקונן. אפשרויות נפוצות, כגון ``optimization_level`` ו-``mitigation_level``, נמצאות ברמה הראשונה. אפשרויות מתקדמות יותר מקובצות לקטגוריות שונות, כגון ``resilience``.

#### ברירות מחדל
אם לא תציין ערך עבור אפשרות, ישמש הערך ברירת המחדל שצוין על ידי השירות.

#### רמת מיתון
פונקציית Circuit של IBM תומכת גם ב-`mitigation_level`. רמת המיתון מציינת כמה דיכוי ומיתון שגיאות להחיל על המשימה. רמות גבוהות יותר מייצרות תוצאות מדויקות יותר, על חשבון זמני עיבוד ארוכים יותר. מידת הפחתת השגיאה תלויה בשיטה המיושמת. רמת המיתון מייצגת בצורה מופשטת את הבחירה המפורטת של שיטות מיתון ודיכוי שגיאות כדי לאפשר למשתמשים לשקול את פשרת העלות/דיוק המתאימה ליישום שלהם. הטבלה הבאה מציגה את השיטות המתאימות לכל רמה.

> **Note:** למרות שהשמות דומים, `mitigation_level` מיישם טכניקות שונות מאלה שבהן משתמש `resilience_level` של Estimator.

בדומה ל-``resilience_level`` ב-Qiskit Runtime Estimator, ``mitigation_level`` מציין קבוצת בסיס של אפשרויות נבחרות. כל אפשרות שתציין ידנית בנוסף לרמת המיתון מוחלת על גבי קבוצת האפשרויות הבסיסית שהוגדרה על ידי רמת המיתון. לכן, באופן עקרוני, אפשר להגדיר את רמת המיתון ל-1 אך לכבות את מיתון המדידה, אם כי אין לעשות זאת.

| **רמת מיתון** | **טכניקה** |
|:-:|:-:|
| 1 [ברירת מחדל] | Dynamical decoupling + measurement twirling + TREX  |
| 2 | רמה 1 + gate twirling + ZNE דרך gate folding |
| 3 | רמה 1 + gate twirling + ZNE דרך PEA |

הדוגמה הבאה מדגימה הגדרת רמת המיתון:

In [7]:
print(f"The result of the submitted job had {len(result)} PUB")
print(
    f"The expectation values measured from this PUB are: \n{result[0].data.evs}"
)
print(f"And the associated metadata is: \n{result[0].metadata}")

The result of the submitted job had 1 PUB
The expectation values measured from this PUB are: 
1.02116704805492
And the associated metadata is: 
{'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32}


#### כל האפשרויות הזמינות
בנוסף ל-`mitigation_level`, פונקציית Circuit של IBM מספקת גם מספר מוגבל של אפשרויות מתקדמות המאפשרות לך לכוונן את פשרת העלות/דיוק. הטבלה הבאה מציגה את כל האפשרויות הזמינות:

| אפשרות | תת-אפשרות | תת-תת-אפשרות | תיאור | בחירות | ברירת מחדל |
|-|-|-|-|-|-|
| default_precision |  |  | הדיוק ברירת המחדל לשימוש עבור כל PUB או קריאת `run()`<br/>שאינה מציינת דיוק משלה.<br/>כל PUB קלט יכול לציין דיוק משלו. אם ל-`run()` ניתן ערך דיוק, הוא ישמש עבור כל ה-PUBs בקריאת `run()` שאינם מציינים דיוק משלהם.  | float > 0 | 0.015625 |
| max_execution_time |  |  | זמן ביצוע מרבי בשניות, המבוסס<br/>על שימוש ב-QPU (לא זמן שעון קיר). שימוש ב-QPU הוא<br/>הזמן שה-QPU מוקדש לעיבוד המשימה שלך. אם משימה חורגת ממגבלת זמן זו, היא תבוטל בכפייה. | מספר שלם של שניות בטווח [1, 10800] |  |
| mitigation_level |  |  | כמה דיכוי ומיתון שגיאות להחיל. עיין בסעיף [רמת מיתון](#mitigation-level) למידע נוסף על השיטות המשמשות בכל רמה. | 1 / 2 / 3 | 1 |
| optimization_level |  |  | כמה אופטימיזציה לבצע על המעגלים. [רמות גבוהות יותר](/guides/set-optimization) מייצרות מעגלים מותאמים יותר, על חשבון זמן טרנספילציה ארוך יותר. | 1 / 2 / 3 | 2 |
| dynamical_decoupling | enable |  | האם לאפשר dynamical decoupling. עיין ב-[טכניקות דיכוי ומיתון שגיאות](/guides/error-mitigation-and-suppression-techniques#dynamical-decoupling) להסבר על השיטה.  | True/False | True |
|  | sequence_type |  | איזו רצף dynamical decoupling להשתמש בו.<br/>* `XX`: שימוש ברצף `tau/2 - (+X) - tau - (+X) - tau/2`<br/>* `XpXm`: שימוש ברצף `tau/2 - (+X) - tau - (-X) - tau/2`<br/>* `XY4`: שימוש ברצף<br/>`tau/2 - (+X) - tau - (+Y) - tau (-X) - tau - (-Y) - tau/2` | 'XX'/'XpXm'/'XY4' | 'XX' |
| twirling | enable_gates |  | האם להחיל twirling של שערי Clifford דו-קיוביטיים. | True/False | False |
|  | enable_measure |  | האם לאפשר twirling של מדידות. | True/False | True |
| resilience | measure_mitigation |  | האם לאפשר שיטת מיתון שגיאות מדידה TREX. עיין ב-[טכניקות דיכוי ומיתון שגיאות](/guides/error-mitigation-and-suppression-techniques#twirled-readout-error-extinction-trex) להסבר על השיטה.  | True/False | True |
|  | zne_mitigation |  | האם להפעיל שיטת מיתון שגיאות Zero Noise Extrapolation. עיין ב-[טכניקות דיכוי ומיתון שגיאות](/guides/error-mitigation-and-suppression-techniques#zero-noise-extrapolation-zne) להסבר על השיטה.  | True/False | False |
|  | zne | amplifier | איזו טכניקה להשתמש בה להגברת רעש. אחת מ: <br/> - `gate_folding` (ברירת מחדל) משתמשת ב-gate folding דו-קיוביטי להגברת רעש. אם גורם הרעש דורש הגברה של תת-קבוצה בלבד של השערים, שערים אלה נבחרים באקראי.<br/><br/> - `gate_folding_front` משתמשת ב-gate folding דו-קיוביטי להגברת רעש. אם גורם הרעש דורש הגברה של תת-קבוצה בלבד של השערים, שערים אלה נבחרים מהחזית של מעגל ה-DAG הממוין טופולוגית.<br/><br/> - `gate_folding_back` משתמשת ב-gate folding דו-קיוביטי להגברת רעש. אם גורם הרעש דורש הגברה של תת-קבוצה בלבד של השערים, שערים אלה נבחרים מהגב של מעגל ה-DAG הממוין טופולוגית.<br/><br/> - `pea` משתמשת בטכניקה הנקראת Probabilistic error amplification‏ (PEA) להגברת רעש. עיין ב-[טכניקות דיכוי ומיתון שגיאות](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-amplification-pea) להסבר על השיטה.  | gate_folding / gate_folding_front / gate_folding_back / pea | gate_folding |
|  |  | noise_factors | גורמי רעש לשימוש בהגברת רעש. | רשימה של floats; כל float >= 1 | (1, 1.5, 2) עבור PEA, ו-(1, 3, 5) בכל מקרה אחר. |
|  |  | extrapolator | גורמי רעש להערכת מודלי ה-fit extrapolation בהם. אפשרות זו אינה משפיעה על הביצוע או התאמת המודל בשום אופן; היא קובעת רק את הנקודות שבהן מוערכים אובייקטי ה-`extrapolator` כדי להחזירם בשדות הנתונים `evs_extrapolated` ו-`stds_extrapolated`. | אחד או יותר מ-`exponential`,`linear`, `double_exponential`,`polynomial_degree_(1 <= k <= 7)` | (`exponential`, `linear`) |
|  | pec_mitigation |  | האם להפעיל שיטת מיתון שגיאות Probabilistic Error Cancellation. עיין ב-[טכניקות דיכוי ומיתון שגיאות](/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) להסבר על השיטה.  | True/False | False |
|  | pec | max_overhead | עומס הדגימה המרבי המותר של המעגל, או `None` ללא מקסימום. | None/ מספר שלם >1 | 100 |

בדוגמה הבאה, הגדרת רמת המיתון ל-1 מכבה בתחילה את מיתון ZNE, אך הגדרת `zne_mitigation` ל-`True` עוקפת את ההגדרה הרלוונטית מ-`mitigation_level`.

In [8]:
job = function.run(
    backend_name="bad_backend_name", pubs=pubs, options=options
)

print(job.result())

QiskitServerlessException: "Traceback (most recent call last):\n  File \"/runner/runner.py\", line 10, in run\n    func = CircuitFunction(**arguments)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/runner/circuit_function/circuit_function.py\", line 87, in __init__\n    self._backend = self._service.backend(\n                    ^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 754, in backend\n    backends = self.backends(name, instance=instance, use_fractional_gates=use_fractional_gates)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/usr/local/lib/python3.11/site-packages/qiskit_ibm_runtime/qiskit_runtime_service.py\", line 497, in backends\n    raise QiskitBackendNotFoundError(\"No backend matches the criteria.\")\nqiskit.providers.exceptions.QiskitBackendNotFoundError: 'No backend matches the criteria.'\n"

## פלטים
הפלט של פונקציית Circuit הוא [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), המכיל שני שדות:

- אובייקט [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) אחד או יותר. ניתן לגשת אליהם ישירות מה-`PrimitiveResult`.

- מטאדאטה ברמת המשימה.

כל `PubResult` מכיל שדה `data` ושדה `metadata`.

- שדה ה-`data` מכיל לפחות מערך של ערכי ציפייה (`PubResult.data.evs`) ומערך של שגיאות תקן (`PubResult.data.stds`). הוא עשוי להכיל גם נתונים נוספים, בהתאם לאפשרויות שבהן נעשה שימוש.

- שדה ה-`metadata` מכיל מטאדאטה ברמת ה-PUB (`PubResult.metadata`).

קטע הקוד הבא מתאר את פורמט ה-`PrimitiveResult` (וה-`PubResult` המשויך).